# 03 — Critical Minerals Production Forecasting

This notebook forecasts future production of key battery and energy-transition minerals using
**statistical time-series models** from the [`statsforecast`](https://github.com/Nixtla/statsforecast) library.

### Models used
| Model | Description |
|-------|-------------|
| **AutoARIMA** | Automatically selects the best ARIMA(p,d,q) order via AIC |
| **AutoETS** | Error–Trend–Seasonality exponential smoothing with automatic component selection |
| **AutoTheta** | Theta method with automatic decomposition (strong baseline for short series) |

All three models are wrapped by `StatsForecast` which vectorises fitting and prediction across
multiple time-series in parallel.

### Optional: Lag-Llama (Foundation Model)
A commented-out section at the end of the notebook shows how to swap in
[Lag-Llama](https://github.com/time-series-foundation-models/lag-llama), a pretrained
transformer for zero-shot and few-shot time-series forecasting, as an additional comparator.

### Data
BGS World Mineral Statistics — annual production figures (1970–2023) for:
- Lithium minerals
- Cobalt (mine)
- Nickel (mine)
- Graphite
- Manganese ore

Top-3 producing countries per mineral (by cumulative production since 2000) are selected,
giving up to **15 individual time-series** to forecast.

In [ ]:
import warnings
import pathlib

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR = pathlib.Path("../data/bgs_data")
CSV_PATH = DATA_DIR / "bgs_critical_minerals_production.csv"

print(f"Data file: {CSV_PATH}")
print(f"Exists   : {CSV_PATH.exists()}")

## 1  Data Preparation

We load the BGS CSV, filter to `statistic_type == 'Production'`, restrict to the five battery
minerals of interest, and identify the top-3 producing countries per mineral based on cumulative
output since 2000.  Series with fewer than 10 non-null observations are dropped to ensure
models have enough history.

In [ ]:
# ── Load & filter ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Raw rows: {len(df_raw):,}")

df = df_raw[df_raw["statistic_type"] == "Production"].copy()
print(f"Production rows: {len(df):,}")

# Coerce year and quantity to numeric, drop anything that can't be parsed
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df = df.dropna(subset=["year", "quantity"])
df["year"] = df["year"].astype(int)

print(f"After numeric coercion: {len(df):,} rows")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")

In [ ]:
# ── Target minerals (exact commodity strings from BGS dataset) ─────────────────
TARGET_MINERALS = [
    "lithium minerals",
    "cobalt, mine",
    "nickel, mine",
    "graphite",
    "manganese ore",
]

# Keep only minerals that actually appear in the data
available = df["commodity"].unique()
minerals  = [m for m in TARGET_MINERALS if m in available]
print("Available target minerals:", minerals)

df_minerals = df[df["commodity"].isin(minerals)].copy()
print(f"Filtered rows: {len(df_minerals):,}")

In [ ]:
# ── Top-3 producers per mineral (by cumulative production since 2000) ──────────
MIN_HISTORY_YEARS = 10

recent = df_minerals[df_minerals["year"] >= 2000]

top_producers: dict[str, list[str]] = {}
for mineral in minerals:
    sub = recent[recent["commodity"] == mineral]
    top3 = (
        sub.groupby("country")["quantity"]
           .sum()
           .nlargest(3)
           .index.tolist()
    )
    top_producers[mineral] = top3
    print(f"  {mineral}: {top3}")

print()

# ── Build forecast_series list ─────────────────────────────────────────────────
forecast_series: list[dict] = []

for mineral, countries in top_producers.items():
    for country in countries:
        mask = (df_minerals["commodity"] == mineral) & (df_minerals["country"] == country)
        series_df = (
            df_minerals[mask][["year", "quantity"]]
            .drop_duplicates("year")
            .sort_values("year")
            .reset_index(drop=True)
        )
        # Require at least MIN_HISTORY_YEARS observations
        if len(series_df) >= MIN_HISTORY_YEARS:
            forecast_series.append(
                {"mineral": mineral, "country": country, "data": series_df}
            )
        else:
            print(f"  SKIP {mineral} | {country} — only {len(series_df)} obs")

print(f"\nTotal series for forecasting: {len(forecast_series)}")
for s in forecast_series:
    d = s["data"]
    print(f"  {s['mineral']} | {s['country']}: {d['year'].min()}–{d['year'].max()} ({len(d)} obs)")

## 2  StatsForecast Modeling

`statsforecast` expects a long-format DataFrame with columns:

| Column | Description |
|--------|-------------|
| `unique_id` | Series identifier (we use `"{mineral}|{country}"`) |
| `ds` | Timestamp of the observation (`pd.Timestamp(year, 1, 1)`) |
| `y` | Numeric target value (production quantity) |

We fit **AutoARIMA**, **AutoETS**, and **AutoTheta** with `season_length=1` (annual data has no
sub-annual seasonality) and forecast **5 years** ahead (`h=5`).

In [ ]:
# ── Build statsforecast long-format DataFrame ──────────────────────────────────
records = []
for s in forecast_series:
    uid = f"{s['mineral']}|{s['country']}"
    for _, row in s["data"].iterrows():
        records.append({
            "unique_id": uid,
            "ds": pd.Timestamp(int(row["year"]), 1, 1),
            "y": float(row["quantity"]),
        })

sf_df = pd.DataFrame(records)
print(f"StatsForecast input shape: {sf_df.shape}")
print(sf_df.head())

In [ ]:
# ── Fit models and forecast ────────────────────────────────────────────────────
FORECAST_HORIZON = 5  # years ahead

models = [
    AutoARIMA(season_length=1),
    AutoETS(season_length=1),
    AutoTheta(season_length=1),
]

sf = StatsForecast(
    models=models,
    freq="YS",          # Year-Start frequency
    n_jobs=-1,          # parallelise across available CPUs
    verbose=True,
)

forecasts_df = sf.forecast(df=sf_df, h=FORECAST_HORIZON)
forecasts_df = forecasts_df.reset_index()
print(f"Forecast output shape: {forecasts_df.shape}")
print(forecasts_df.head(10))

## 3  Forecast Visualisation

For up to 6 series we plot:

- **Historical data** — solid blue line
- **AutoARIMA forecast** — dashed red line
- **AutoETS forecast** — dashed green line
- **AutoTheta forecast** — dashed orange line

Each subplot covers one `mineral | country` combination.

In [ ]:
# ── Helper: retrieve historical series for a unique_id ─────────────────────────
def get_history(uid: str) -> pd.DataFrame:
    """Return the historical (ds, y) rows for a given unique_id."""
    return sf_df[sf_df["unique_id"] == uid].sort_values("ds")


def get_forecast(uid: str) -> pd.DataFrame:
    """Return the forecast rows for a given unique_id."""
    return forecasts_df[forecasts_df["unique_id"] == uid].sort_values("ds")


# ── Build subplot grid ─────────────────────────────────────────────────────────
MODEL_COLS   = ["AutoARIMA", "AutoETS", "AutoTheta"]
MODEL_COLORS = {"AutoARIMA": "red", "AutoETS": "green", "AutoTheta": "orange"}

plot_series = [s["mineral"] + "|" + s["country"] for s in forecast_series][:6]
n_plots     = len(plot_series)
n_cols      = 2
n_rows      = (n_plots + n_cols - 1) // n_cols

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=plot_series,
    shared_xaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

for idx, uid in enumerate(plot_series):
    row = idx // n_cols + 1
    col = idx % n_cols + 1

    hist     = get_history(uid)
    fc       = get_forecast(uid)
    show_leg = idx == 0  # only add legend entries once

    # Historical
    fig.add_trace(
        go.Scatter(
            x=hist["ds"], y=hist["y"],
            mode="lines+markers",
            line=dict(color="steelblue", width=2),
            marker=dict(size=4),
            name="Historical",
            legendgroup="Historical",
            showlegend=show_leg,
        ),
        row=row, col=col,
    )

    # Model forecasts
    for model in MODEL_COLS:
        if model not in fc.columns:
            continue
        # Connect last historical point to first forecast point for visual continuity
        last_hist = hist.iloc[[-1]][["ds", "y"]].rename(columns={"y": model})
        fc_ext    = pd.concat([last_hist, fc[["ds", model]]], ignore_index=True)

        fig.add_trace(
            go.Scatter(
                x=fc_ext["ds"], y=fc_ext[model],
                mode="lines+markers",
                line=dict(color=MODEL_COLORS[model], width=2, dash="dash"),
                marker=dict(size=5, symbol="diamond"),
                name=model,
                legendgroup=model,
                showlegend=show_leg,
            ),
            row=row, col=col,
        )

fig.update_layout(
    title_text="Critical Minerals Production Forecasts (5-year horizon)",
    title_font_size=18,
    height=350 * n_rows,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template="plotly_white",
)
fig.show()

## 4  Optional: Lag-Llama Foundation Model

[Lag-Llama](https://github.com/time-series-foundation-models/lag-llama) is a **pretrained
transformer** for univariate probabilistic time-series forecasting.  Unlike the statistical
models above, it requires no per-series fitting — it can forecast in a zero-shot manner or
be fine-tuned with a few epochs on the target domain.

The cell below is **commented out**.  To activate it:
1. Uncomment the `%pip install` line and run it.
2. Download the Lag-Llama checkpoint from Hugging Face (see comments in the cell).
3. Uncomment and run the remaining code.

> **Note:** Lag-Llama requires PyTorch.  A GPU is not required for small datasets like this
> one but will significantly speed up inference on larger corpora.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OPTIONAL — Lag-Llama Foundation Model (all code is COMMENTED OUT)         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Step 1: install dependencies

# Step 2: download checkpoint
# from huggingface_hub import hf_hub_download
# ckpt_path = hf_hub_download(
#     repo_id="time-series-foundation-models/Lag-Llama",
#     filename="lag-llama.ckpt",
# )

# Step 3: build GluonTS ListDataset from forecast_series
# from gluonts.dataset.common import ListDataset
# import torch
#
# gluon_train = ListDataset(
#     [
#         {
#             "start": pd.Period(str(s["data"]["year"].min()), freq="Y"),
#             "target": s["data"]["quantity"].values.astype(float),
#             "item_id": f"{s['mineral']}|{s['country']}",
#         }
#         for s in forecast_series
#     ],
#     freq="Y",
# )

# Step 4: instantiate and run LagLlamaEstimator
# from lag_llama.gluon.estimator import LagLlamaEstimator
#
# estimator = LagLlamaEstimator(
#     ckpt_path=ckpt_path,
#     prediction_length=FORECAST_HORIZON,
#     context_length=32,
#     n_parallel_samples=100,
#     device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
#     batch_size=8,
#     num_parallel_samples=100,
# )
#
# # Zero-shot prediction (no training required)
# predictor  = estimator.create_predictor(transformation=estimator.create_transformation())
# lag_fc     = list(predictor.predict(gluon_train))
#
# # Extract median forecasts
# lag_results = []
# for i, fc_entry in enumerate(lag_fc):
#     item_id  = fc_entry.item_id
#     start_yr = fc_entry.start_date.year
#     median   = fc_entry.quantile(0.5)
#     for j, val in enumerate(median):
#         lag_results.append({
#             "unique_id": item_id,
#             "ds": pd.Timestamp(start_yr + j, 1, 1),
#             "LagLlama": val,
#         })
#
# lag_df = pd.DataFrame(lag_results)
# print(lag_df.head())

print("Lag-Llama cell is commented out. Uncomment to use the foundation model.")

## 5  Model Comparison via Cross-Validation

We use `StatsForecast.cross_validation` with a rolling-window scheme:
- **`h=3`** — 3-year forecast horizon per window
- **`step_size=1`** — windows shift by 1 year
- **`n_windows=2`** — 2 evaluation windows

Performance is measured with **Mean Absolute Percentage Error (MAPE)**, aggregated across
all series and both windows.

In [ ]:
# ── Cross-validation ───────────────────────────────────────────────────────────
cv_df = sf.cross_validation(
    df=sf_df,
    h=3,
    step_size=1,
    n_windows=2,
)
cv_df = cv_df.reset_index()
print(f"CV output shape: {cv_df.shape}")
print(cv_df.head())

In [ ]:
# ── Compute MAPE per model ─────────────────────────────────────────────────────
def mape(actual: pd.Series, predicted: pd.Series) -> float:
    """Mean Absolute Percentage Error, ignoring zeros in actual."""
    mask = actual != 0
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)


mape_scores: dict[str, float] = {}
for model in MODEL_COLS:
    if model in cv_df.columns:
        mape_scores[model] = mape(cv_df["y"], cv_df[model])

mape_df = pd.DataFrame(
    [{"Model": m, "MAPE (%)": v} for m, v in mape_scores.items()]
).sort_values("MAPE (%)")

print("\nModel MAPE scores:")
print(mape_df.to_string(index=False))

In [ ]:
# ── Bar chart: model comparison ────────────────────────────────────────────────
bar_colors = [MODEL_COLORS.get(m, "steelblue") for m in mape_df["Model"]]

fig_cv = go.Figure(
    go.Bar(
        x=mape_df["Model"],
        y=mape_df["MAPE (%)"],
        marker_color=bar_colors,
        text=mape_df["MAPE (%)"].round(1).astype(str) + "%",
        textposition="outside",
    )
)

fig_cv.update_layout(
    title_text="Model Comparison — Mean Absolute Percentage Error (MAPE)",
    title_font_size=16,
    xaxis_title="Model",
    yaxis_title="MAPE (%)",
    yaxis=dict(rangemode="tozero"),
    template="plotly_white",
    height=450,
    width=650,
)
fig_cv.show()

## Summary

| Step | Description |
|------|-------------|
| Data prep | Filtered BGS production data, selected top-3 producers per mineral (since 2000), required ≥10 years history |
| Modelling | AutoARIMA, AutoETS, AutoTheta via `statsforecast`; annual frequency, 5-year horizon |
| Evaluation | 2-window cross-validation (h=3, step=1); MAPE used as error metric |
| Extension | Lag-Llama zero-shot foundation model code provided (commented out) |

Lower MAPE indicates better point-forecast accuracy.  For policy applications, consider also
examining prediction intervals (available via `level` parameter in `sf.forecast`) to understand
supply uncertainty ranges.